# 가전별 ON 비율 검증

> **목적**: `postprocessor.py`의 `ALWAYS_ON_IDX` 대상(냉장고 2종) 타당성 확인  
> GCS 실데이터 기준으로 가전별 ON 비율을 출력하여 항상 ON 고정이 적절한지 검증한다.

**실행 순서**: 1 → 2 → 3 → 4

## 1. 환경 설치 & GCS 인증

In [ ]:
!pip install -q gcsfs pyarrow pywavelets gudhi

from google.colab import auth
auth.authenticate_user()
print("GCS 인증 완료")

## 2. Repo 클론 & 경로 설정

In [ ]:
import os, sys

REPO_DIR = "/content/ax_nilm"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dhwang0803-glitch/ax_nilm {REPO_DIR} -q
    print("클론 완료")
else:
    !git -C {REPO_DIR} pull -q && echo "최신화 완료"

SRC_DIR = f"{REPO_DIR}/nilm-engine/src"
CFG_DIR = f"{REPO_DIR}/nilm-engine/config"
for d in [SRC_DIR, f"{REPO_DIR}/nilm-engine/scripts"]:
    if d not in sys.path:
        sys.path.insert(0, d)
print("경로 설정 완료")

## 3. 설정

In [ ]:
import yaml
import gcsfs

gcs = gcsfs.GCSFileSystem()

BUCKET_PREFIX = "ax-nilm-data-dhwang0803-us/nilm/training_dev10"
LABEL_PATH    = "ax-nilm-data-dhwang0803-us/nilm/labels/training.parquet"

# 검증 대상 house — 학습에 쓰지 않은 test house로 실측 ON 비율 확인
# train: house_011/015/016/017/033/039/054/063  val: house_049  test: house_067
CHECK_HOUSES = ["house_067"]

with open(f"{CFG_DIR}/dataset.yaml") as f:
    DATASET_CFG = yaml.safe_load(f)

ws = DATASET_CFG["window"]["size"]
st = DATASET_CFG["window"]["stride"]
ec = DATASET_CFG["window"].get("event_context")
ss = DATASET_CFG["window"].get("steady_stride")

print(f"window={ws}, stride={st}, event_context={ec}, steady_stride={ss}")
print(f"검증 house: {CHECK_HOUSES}")

## 4. ON 비율 집계 & 출력

In [ ]:
import numpy as np
from acquisition.gcs_loader import GCSNILMDataset
from classifier.label_map import APPLIANCE_LABELS, APPLIANCE_LABELING

ds = GCSNILMDataset(
    CHECK_HOUSES,
    gcs_fs=gcs,
    bucket_prefix=BUCKET_PREFIX,
    label_path=LABEL_PATH,
    window_size=ws,
    stride=st,
    event_context=ec,
    steady_stride=ss,
    fit_scaler=False,
)

all_on    = np.zeros(len(APPLIANCE_LABELS), dtype=np.int64)
all_total = np.zeros(len(APPLIANCE_LABELS), dtype=np.int64)

for _agg, _tgt, on_mask, _validity in ds._segments:  # on_mask: (N_APP, n_samples) bool
    all_on    += on_mask.sum(axis=1)
    all_total += on_mask.shape[1]

print(f"{'가전':<22} {'ON비율':>8}  {'threshold_kind':<14}  비고")
print("-" * 72)
for i, name in enumerate(APPLIANCE_LABELS):
    if all_total[i] == 0:
        continue
    ratio = all_on[i] / all_total[i]
    kind  = APPLIANCE_LABELING.get(name, {}).get("threshold_kind", "?")

    if kind == "always_on":
        note = "항상ON 고정 대상 ✓" if ratio > 0.90 else "⚠ 실측이 낮음 — 재확인 필요"
    elif ratio > 0.90:
        note = "← ON 비율 높음, always_on 고정 아님 (정상)"
    else:
        note = ""

    print(f"{name:<22} {ratio:>8.3f}  {kind:<14}  {note}")